In [12]:
from pyspark.sql.functions import variance
from pyspark.sql.types import TimestampType, StringType
import pandas as pd
numeric_cols = [f.name for f in df_master.schema.fields
                if not isinstance(f.dataType, (TimestampType, StringType))]
variance_exprs = [variance(col(c)).alias(c) for c in numeric_cols]
variance_result = df_master.agg(*variance_exprs).collect()[0].asDict()
variance_pdf = pd.DataFrame(list(variance_result.items()), columns=["Feature", "Variance"])
variance_pdf = variance_pdf.sort_values(by="Variance", ascending=True)
print("--- BẢNG PHƯƠNG SAI CÁC ĐẶC TRƯNG ---")
print(variance_pdf.to_string(index=False))

--- BẢNG PHƯƠNG SAI CÁC ĐẶC TRƯNG ---
        Feature  Variance
            LPS  0.003408
Pressure_switch  0.008490
Caudal_impulses  0.058938
         Towers  0.073727
      Oil_level  0.086658
    DV_electric  0.134815
           COMP  0.136460
            MPG  0.139335
    DV_pressure  0.146231
     Reservoirs  0.407436
            TP3  0.408443
            dow  3.880615
          month  4.056395
  Motor_current  5.299450
            TP2 10.568544
 Pressure_Delta 11.027454
             H1 11.110219
Oil_temperature 42.461659
           hour 43.652590


In [13]:
df_master = df_master.drop("LPS", "Pressure_switch")
df_master = df_master.persist(StorageLevel.MEMORY_AND_DISK)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.stat import Correlation
cols_to_check = ["TP3", "Reservoirs", "TP2", "Motor_current", "Oil_temperature", "Pressure_Delta"]
assembler_corr = VectorAssembler(inputCols=cols_to_check, outputCol="corr_features")
df_corr = assembler_corr.transform(df_master).select("corr_features")
corr_matrix = Correlation.corr(df_corr, "corr_features").head()[0]
corr_pdf = pd.DataFrame(corr_matrix.toArray(), columns=cols_to_check, index=cols_to_check)
print("--- MA TRẬN TƯƠNG QUAN GIỮA CÁC BIẾN ANALOG ---")
print(corr_pdf.to_string())

--- MA TRẬN TƯƠNG QUAN GIỮA CÁC BIẾN ANALOG ---
                      TP3  Reservoirs       TP2  Motor_current  Oil_temperature  Pressure_Delta
TP3              1.000000    0.999993 -0.011161       0.413756         0.401616        0.203142
Reservoirs       0.999993    1.000000 -0.012403       0.412691         0.401647        0.204359
TP2             -0.011161   -0.012403  1.000000       0.697480         0.250710       -0.981355
Motor_current    0.413756    0.412691  0.697480       1.000000         0.528739       -0.603487
Oil_temperature  0.401616    0.401647  0.250710       0.528739         1.000000       -0.168234
Pressure_Delta   0.203142    0.204359 -0.981355      -0.603487        -0.168234        1.000000


In [14]:
df_master = df_master.drop("LPS", "Pressure_switch", "TP3", "TP2")
df_master = df_master.persist(StorageLevel.MEMORY_AND_DISK)
print(f"Tổng số lượng đặc trưng: {len(df_master.columns)}")
print("Danh sách đặc trưng:", df_master.columns)

Tổng số lượng đặc trưng: 17
Danh sách đặc trưng: ['_c0', 'timestamp', 'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current', 'COMP', 'DV_electric', 'Towers', 'MPG', 'Oil_level', 'Caudal_impulses', 'hour', 'month', 'dow', 'Pressure_Delta']


In [15]:
from pyspark.ml.feature import PCA, VectorAssembler, StandardScaler
physical_features = [
    'H1', 'DV_pressure', 'Reservoirs', 'Oil_temperature', 'Motor_current',
    'COMP', 'DV_electric', 'Towers', 'MPG', 'Oil_level', 'Caudal_impulses', 'Pressure_Delta'
]
assembler = VectorAssembler(inputCols=physical_features, outputCol="features_vec")
df_vec = assembler.transform(df_master.drop('_c0', 'timestamp', 'hour', 'month', 'dow'))
scaler = StandardScaler(inputCol="features_vec", outputCol="features_scaled", withStd=True, withMean=True)
scaler_model = scaler.fit(df_vec)
df_scaled = scaler_model.transform(df_vec)
pca = PCA(k=6, inputCol="features_scaled", outputCol="pca_features")
pca_model = pca.fit(df_scaled)
import pandas as pd
loadings = pca_model.pc.toArray()
df_loadings = pd.DataFrame(loadings, index=physical_features, columns=[f"PC{i+1}" for i in range(6)])
print(df_loadings.abs().sort_values(by="PC1", ascending=False))

                      PC1       PC2       PC3       PC4       PC5       PC6
COMP             0.396725  0.055795  0.038049  0.040627  0.075099  0.075264
MPG              0.395260  0.038420  0.083330  0.040342  0.063078  0.104201
DV_electric      0.392106  0.041706  0.005835  0.044980  0.083645  0.171454
H1               0.389559  0.153208  0.000535  0.016277  0.067829  0.135629
Pressure_Delta   0.386438  0.140272  0.007150  0.014164  0.059472  0.144258
Motor_current    0.298056  0.397668  0.091696  0.118214  0.142525  0.104983
Towers           0.285620  0.040054  0.169499  0.038405  0.052071  0.932502
DV_pressure      0.206099  0.003192  0.330316  0.352130  0.669190  0.091376
Oil_temperature  0.131530  0.566817  0.138474  0.235935  0.296641  0.011629
Reservoirs       0.028976  0.651347  0.094369  0.252190  0.272765  0.086374
Caudal_impulses  0.020750  0.127181  0.673839  0.460765  0.544975  0.112018
Oil_level        0.008800  0.170469  0.602608  0.723339  0.212163  0.076156


In [16]:
import numpy as np
import pandas as pd
from pyspark.ml.feature import PCA
pca_objective = PCA(k=12, inputCol="features_scaled", outputCol="pca_features_obj")
pca_model_obj = pca_objective.fit(df_scaled)
explained_variance = pca_model_obj.explainedVariance
cum_variance = np.cumsum(explained_variance)
df_variance = pd.DataFrame({
    "Component": range(1, 13),
    "Explained Variance": explained_variance,
    "Cumulative Variance": cum_variance
})
print("--- BẢNG PHƯƠNG SAI TÍCH LŨY (PCA) ---")
print(df_variance.to_string(index=False))

--- BẢNG PHƯƠNG SAI TÍCH LŨY (PCA) ---
 Component  Explained Variance  Cumulative Variance
         1            0.509622             0.509622
         2            0.145008             0.654630
         3            0.090966             0.745596
         4            0.079281             0.824877
         5            0.074018             0.898895
         6            0.044118             0.943014
         7            0.031416             0.974430
         8            0.015923             0.990353
         9            0.005253             0.995606
        10            0.002985             0.998591
        11            0.001055             0.999646
        12            0.000354             1.000000


In [17]:
import numpy as np
import pandas as pd
df_loadings['Score'] = np.sqrt((df_loadings**2).sum(axis=1))
print("--- ĐIỂM ĐÓNG GÓP CỦA CÁC ĐẶC TRƯNG (PCA MAGNITUDE) ---")
print(df_loadings[['Score']].sort_values(by='Score', ascending=True))

--- ĐIỂM ĐÓNG GÓP CỦA CÁC ĐẶC TRƯNG (PCA MAGNITUDE) ---
                    Score
COMP             0.418219
MPG              0.425576
Pressure_Delta   0.440011
DV_electric      0.440382
H1               0.445522
Motor_current    0.548357
Oil_temperature  0.708203
Reservoirs       0.761222
DV_pressure      0.855427
Oil_level        0.983008
Towers           0.992804
Caudal_impulses  0.996250


In [18]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator
from pyspark import StorageLevel
import pandas as pd
final_features = [
    'Caudal_impulses', 'Towers', 'Oil_level', 'DV_pressure',
    'Reservoirs', 'Oil_temperature', 'Motor_current'
]
assembler_final = VectorAssembler(inputCols=final_features, outputCol="features_final")
scaler_final = StandardScaler(inputCol="features_final", outputCol="features_scaled_final", withStd=True, withMean=True)
pipeline_final = Pipeline(stages=[assembler_final, scaler_final])
model_final = pipeline_final.fit(df_master)
df_final_kmeans = model_final.transform(df_master).select("features_scaled_final").persist(StorageLevel.MEMORY_AND_DISK)
k_values = range(2, 9)
silhouette_scores = []
print(f"{'K':<5} | {'Silhouette Score':<20}")
print("-" * 30)
for k in k_values:
    kmeans = KMeans(featuresCol="features_scaled_final", k=k, seed=42)
    model = kmeans.fit(df_final_kmeans)
    predictions = model.transform(df_final_kmeans)
    evaluator = ClusteringEvaluator(
        featuresCol="features_scaled_final",
        metricName="silhouette",
        distanceMeasure="squaredEuclidean"
    )
    score = evaluator.evaluate(predictions)
    silhouette_scores.append(score)
    print(f"{k:<5} | {score:.4f}")
df_results = pd.DataFrame({"K": list(k_values), "Silhouette": silhouette_scores})

K     | Silhouette Score    
------------------------------
2     | 0.6287
3     | 0.4556
4     | 0.5410
5     | 0.6173
6     | 0.6781
7     | 0.5308
8     | 0.5355
